In [ ]:
# =============================================================
# PEAP (Kirchhoff exact) on Evoked Theta  [new RAIN_IRC data + FIR-175 GT, matched to BOB/EPN]
# =============================================================
# Predictor (UNCHANGED): raw -> Burg AR -> AR-extrapolate pad -> FIR(4-8 Hz) -> Hilbert.
# Reads the SAME npz BOB/EPN read (RAIN_IRC/data splits/onset, native 300 Hz); channel =
# best_ch_idx, test set = the npz split array, GT = 4-8 Hz zero-phase FIR-175 (byte-identical
# taps to BOB). PEAP's sample-denominated params are scaled 600 Hz -> 300 Hz to preserve
# their durations (AR 250->125, context 980->490, pad 290->145, FIR 351->175 taps).
# Single GT (the Butterworth GT is dropped) and RAW test errors (no bias correction, like
# EPN) so it is scored identically to BOB/EPN. Output: one model-tagged JSON, BOB schema.

import numpy as np
import os, time, json, gc, glob, warnings
from scipy.signal import firwin, filtfilt, hilbert, welch
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

STIM_DIR = '/content/drive/MyDrive/RAIN_IRC/data splits/onset'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/results/PEAP_onset'
os.makedirs(OUTDIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(STIM_DIR, '*.npz')))
assert files, f"no npz found in {STIM_DIR}"
_d0 = np.load(files[0], allow_pickle=True)
FS = float(_d0['fs']); PRE_SAMP = int(_d0['onset_in_epoch']); _d0.close()

EVAL_TIMES_S    = [0.3, 0.4, 0.5, 0.6, 0.7]
EVAL_TIMES_SAMP = [PRE_SAMP + int(round(t * FS)) for t in EVAL_TIMES_S]
THETA_SEARCH_LO, THETA_SEARCH_HI = 4.0, 8.0

# --- GT: 4-8 Hz zero-phase FIR-175 (filtfilt), identical taps to BOB ---
GT_BAND        = [4.0, 8.0]
GT_FIR_NUMTAPS = 175                              # 300 Hz (half of 351 @ 600 Hz)
gt_fir_b = firwin(GT_FIR_NUMTAPS, GT_BAND, fs=FS, window='hamming', pass_zero=False)
GT_HALF_S = (len(gt_fir_b) // 2) / FS             # ~0.29 s acausal reach

# --- PEAP fixed specs, scaled 600 Hz -> 300 Hz to preserve durations ---
PEAP_AR_ORDER     = 130                           # 250 @ 600 Hz   (see note: file header said 130)
PEAP_CONTEXT_SAMP = 980                           # 980 @ 600 Hz   (~1.63 s)
PEAP_PAD_SAMP     = 87                          # 290 @ 600 Hz   (~0.48 s)
PEAP_FIR_NUMTAPS  = 175                           # 351 @ 600 Hz
PEAP_FIR_BAND     = [4.0, 8.0]
peap_fir_b = firwin(PEAP_FIR_NUMTAPS, PEAP_FIR_BAND, fs=FS, window='hamming', pass_zero=False)

print(f"\n{'='*84}")
print("PEAP (Kirchhoff exact) on Evoked Theta  —  new RAIN_IRC 300 Hz data, FIR-175 GT")
print(f"{'='*84}")
print(f"  FS={FS:.0f} Hz | t=0 at sample {PRE_SAMP}")
print(f"  GT (FIR): {len(gt_fir_b)} taps @ {FS:.0f} Hz, {GT_BAND} Hz, half-length {GT_HALF_S:.3f}s")
print(f"  PEAP: raw -> Burg AR({PEAP_AR_ORDER}) -> pad {PEAP_PAD_SAMP} "
      f"-> FIR({PEAP_FIR_NUMTAPS}, {PEAP_FIR_BAND}) -> Hilbert @ {FS:.0f} Hz")
print(f"  Data: {STIM_DIR}  ({len(files)} subjects)")
print(f"  Output: {OUTDIR}")
inside = [t for t in EVAL_TIMES_S if t < GT_HALF_S]
print("  onset-smear check: " + (f"*** {inside}s inside GT half-length {GT_HALF_S:.3f}s"
      if inside else f"OK (earliest {min(EVAL_TIMES_S)}s >= half-length {GT_HALF_S:.3f}s)"))


# --- Burg AR ---  [UNCHANGED]
def burg_ar(x, order):
    x = x.astype(np.float64); n = len(x); order = int(order)
    if n <= order + 1: return np.zeros(order, dtype=np.float64)
    ef, eb = x.copy(), x.copy(); a = np.zeros(order, dtype=np.float64)
    for m in range(order):
        ef_s, eb_s = ef[m+1:n], eb[m:n-1]; L = min(len(ef_s), len(eb_s))
        if L <= 0: break
        ef_s, eb_s = ef_s[:L].copy(), eb_s[:L].copy()
        num = -2.0 * np.dot(ef_s, eb_s); den = np.dot(ef_s, ef_s) + np.dot(eb_s, eb_s)
        if den < 1e-12: break
        k = np.clip(num / den, -0.9999, 0.9999)
        a_new = np.zeros(m + 1, dtype=np.float64); a_new[m] = k
        if m > 0: a_new[:m] = a[:m] + k * a[m-1::-1]
        a[:m + 1] = a_new
        ef[m+1:m+1+L] = ef_s + k * eb_s; eb[m+1:m+1+L] = eb_s + k * ef_s; n = m + 1 + L
    return -a

def ar_extrapolate(x, coeffs, n_steps):
    p = len(coeffs); buf = list(x.astype(np.float64)); x_max = np.max(np.abs(x)) * 10.0
    for _ in range(n_steps):
        if len(buf) < p: buf.append(buf[-1]); continue
        pred = sum(coeffs[k] * buf[-k-1] for k in range(p))
        if abs(pred) > x_max: break
        buf.append(pred)
    return np.array(buf, dtype=np.float64)

# --- PEAP phase prediction (NaN on failure, never silent 0) ---  [UNCHANGED logic, native FS]
def peap_predict(raw_1d, eval_samp):
    ctx_start = max(0, eval_samp - PEAP_CONTEXT_SAMP + 1)
    context = raw_1d[ctx_start:eval_samp + 1].astype(np.float64)
    ctx_len = len(context)
    ar_ord = min(PEAP_AR_ORDER, ctx_len // 3)
    if ctx_len < ar_ord + 5 or np.std(context) < 1e-10:
        return np.nan
    coeffs = burg_ar(context, ar_ord)
    compound = ar_extrapolate(context, coeffs, PEAP_PAD_SAMP)
    if len(compound) < 3 * len(peap_fir_b) + 1:
        return np.nan
    try:
        padlen = min(3 * len(peap_fir_b), len(compound) - 1)
        filtered = filtfilt(peap_fir_b, [1.0], compound, padlen=padlen)
    except Exception:
        return np.nan
    target_idx = ctx_len - 1
    if target_idx >= len(filtered):
        return np.nan
    return float(np.angle(hilbert(filtered)[target_idx]))

# --- GT phase (acausal FIR-175 on the 300 Hz epoch) ---
def gt_phase_fir(epoch):
    try:
        padlen = min(3 * len(gt_fir_b), len(epoch) - 1)
        f = filtfilt(gt_fir_b, [1.0], epoch.astype(np.float64), padlen=padlen)
        return np.angle(hilbert(f))
    except Exception:
        return np.full(len(epoch), np.nan)

def circ(a, b):
    return np.angle(np.exp(1j * (a - b)))

# ---- metrics: BOB-identical (MACE, PLV, CSD, Acc%, bias); NaN errors dropped ----
def errors_to_metrics(errors):
    errors = np.asarray(errors, dtype=float); errors = errors[np.isfinite(errors)]
    if len(errors) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    mean_abs = float(np.mean(np.abs(errors))); R = float(np.abs(np.mean(np.exp(1j * errors))))
    return {'mace': float(np.degrees(mean_abs)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),   # circular SD (deg)
            'acc': float((1.0 - mean_abs / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * errors))))),
            'n_samples': int(len(errors))}

def itpc_at(ph_list, trials, esamp):
    """GT-phase ITPC across trials at one eval sample (BOB-identical basis)."""
    vals = [ph_list[tr][esamp] for tr in trials
            if esamp < len(ph_list[tr]) and np.isfinite(ph_list[tr][esamp])]
    if len(vals) < 2: return float('nan')
    return float(np.abs(np.mean(np.exp(1j * np.array(vals)))))

def estimate_theta_peak(raw_1ch, onset_samp):
    seg = raw_1ch[:, onset_samp:]
    if seg.shape[1] < 16: return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= THETA_SEARCH_LO) & (f <= THETA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan

Mounted at /content/drive

PEAP (Kirchhoff exact) on Evoked Theta  —  new RAIN_IRC 300 Hz data, FIR-175 GT
  FS=300 Hz | t=0 at sample 300
  GT (FIR): 175 taps @ 300 Hz, [4.0, 8.0] Hz, half-length 0.290s
  PEAP: raw -> Burg AR(130) -> pad 87 -> FIR(175, [4.0, 8.0]) -> Hilbert @ 300 Hz
  Data: /content/drive/MyDrive/RAIN_IRC/data splits/onset  (20 subjects)
  Output: /content/drive/MyDrive/RAIN_IRC/results/PEAP_onset
  onset-smear check: OK (earliest 0.3s >= half-length 0.290s)


In [ ]:
# =============================================================
# Main loop   [PEAP predictor UNCHANGED; evaluated on the npz TEST split]
# =============================================================
print(f"{'Subj':<14s}{'N':>5s}{'MACE':>8s}{'CSD':>8s}{'Acc%':>7s}{'PLV':>7s}{'Bias':>7s}{'SNR':>6s}{'ITPC':>6s}{'Peak':>6s}{'fail%':>7s}")
print("-" * 98)

results = []
for fp in files:
    t0 = time.time(); d = np.load(fp, allow_pickle=True); name = os.path.basename(fp)[:-4]
    if float(d['fs']) != FS or int(d['onset_in_epoch']) != PRE_SAMP:
        print(f"{name}: geometry mismatch, skipping"); d.close(); continue
    epochs = d['epochs']; best = int(d['best_ch_idx']); ch_name = str(d['best_ch_name'])
    split_arr = np.array([str(x) for x in d['split']]); theta_snr = float(d['ch_theta_snr'][best])
    raw = epochs[:, best, :].astype(np.float64)
    n_trials, n_ch, epoch_len = epochs.shape
    test_trials = np.where(split_arr == 'test')[0]

    # GT phase per trial (FIR-175, same as BOB)
    ph_fir = [gt_phase_fir(raw[tr]) for tr in range(n_trials)]

    # per-subject theta descriptors (GT-phase ITPC at eval pts on TEST + Welch peak)
    itpc_tp = {t: (itpc_at(ph_fir, test_trials, es) if es < epoch_len else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values())))
    f_peak = estimate_theta_peak(raw, PRE_SAMP)

    # PEAP causal phase prediction on TEST, per eval point (RAW errors, NaN dropped + counted)
    per_tp = {t: [] for t in EVAL_TIMES_S}
    n_eval = n_fail = 0
    for tr in test_trials:
        for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP):
            if es >= epoch_len: continue
            n_eval += 1
            est = peap_predict(raw[tr], es)
            gf = ph_fir[tr][es]
            if not (np.isfinite(est) and np.isfinite(gf)):
                n_fail += 1; continue
            per_tp[t].append(circ(est, gf))
    per_tp = {t: np.array(v) for t, v in per_tp.items()}
    fail_pct = 100.0 * n_fail / max(n_eval, 1)

    pooled = (np.concatenate([per_tp[t] for t in EVAL_TIMES_S])
              if any(len(per_tp[t]) for t in EVAL_TIMES_S) else np.array([]))
    met = errors_to_metrics(pooled)
    per_tp_met = {t: errors_to_metrics(per_tp[t]) for t in EVAL_TIMES_S}

    dt = time.time() - t0
    pk = f"{f_peak:.2f}" if np.isfinite(f_peak) else " n/a"
    print(f"{name:<14s}{met['n_samples']:>5d}{met['mace']:>7.1f}°{met['csd']:>7.1f}°"
          f"{met['acc']:>6.1f}%{met['plv']:>7.3f}{met['bias']:>+6.1f}°"
          f"{theta_snr:>6.2f}{itpc_mean:>6.3f}{pk:>6s}{fail_pct:>6.1f}%  [{dt:.0f}s]")

    results.append({
        'name': name, 'best_ch': ch_name, 'n_trials': int(n_trials),
        'n_test_samples': met['n_samples'], 'fail_pct': fail_pct, 'theta_snr': theta_snr,
        'theta_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'theta_itpc_mean': itpc_mean,
        'theta_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'metrics_pooled': met,
        'metrics_per_timepoint': {str(t): per_tp_met[t] for t in EVAL_TIMES_S},
        'errors_per_timepoint_rad': {str(t): [float(x) for x in per_tp[t]] for t in EVAL_TIMES_S}})
    d.close(); del epochs, raw, ph_fir; gc.collect()

print(f"\n{len(results)} subjects done")

Subj              N    MACE     CSD   Acc%    PLV   Bias   SNR  ITPC  Peak  fail%
--------------------------------------------------------------------------------------------------
Subject10001    150   51.1°   64.7°  71.6%  0.528  -4.7°  2.56 0.097  4.03  40.0%  [3s]
Subject1001     150   49.1°   62.8°  72.7%  0.548  -8.2°  2.59 0.162  4.03  40.0%  [3s]
Subject11002    150   50.5°   63.9°  71.9%  0.537  -8.8°  2.04 0.128  4.03  40.0%  [5s]
Subject12001    150   54.6°   71.4°  69.6%  0.460  -0.7°  1.93 0.179  4.03  40.0%  [6s]
Subject13001    150   42.6°   54.7°  76.3%  0.634  -2.0°  2.65 0.072  6.88  40.0%  [5s]
Subject14001    150   60.1°   77.3°  66.6%  0.403  -1.5°  2.41 0.169  4.03  40.0%  [5s]
Subject15001    150   60.8°   78.2°  66.2%  0.394  +5.9°  2.36 0.146  4.03  40.0%  [5s]
Subject16001    150   50.6°   64.4°  71.9%  0.531  -0.6°  1.74 0.192  4.03  40.0%  [5s]
Subject17001    150   52.8°   66.7°  70.7%  0.508  +0.9°  2.45 0.205  4.03  40.0%  [5s]
Subject18002    150   51.7°

In [ ]:
# =============================================================
# SUMMARY + SAVE  (complete model-tagged JSON, BOB-parallel)
# =============================================================
valid = [r for r in results if r['n_test_samples'] >= 10]
n = len(valid)
M  = np.array([r['metrics_pooled']['mace'] for r in valid])
P  = np.array([r['metrics_pooled']['plv'] for r in valid])
Cd = np.array([r['metrics_pooled']['csd'] for r in valid])
A  = np.array([r['metrics_pooled']['acc'] for r in valid])
SNR = np.array([r['theta_snr'] for r in valid])
IT  = np.array([r['theta_itpc_mean'] for r in valid])

print('=' * 72); print('PEAP THETA — group summary'); print('=' * 72)
print(f"  n={n} participants (>=10 test samples)")
print(f"  MACE {M.mean():.1f}°±{M.std():.1f}   PLV {P.mean():.3f}   CSD {Cd.mean():.1f}°   Acc {A.mean():.1f}%")
print(f"  mean failure rate {np.mean([r['fail_pct'] for r in results]):.1f}%")

print('\n--- Per-timepoint ---')
print(f"{'ms':>5}{'MACE':>9}{'PLV':>8}{'CSD':>8}{'Acc':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    g = lambda k: np.array([r['metrics_per_timepoint'][str(t)][k] for r in valid])
    ti = np.array([r['theta_itpc_per_timepoint'][str(t)] for r in valid])
    print(f"{int(t*1000):>5}{g('mace').mean():>8.1f}°{g('plv').mean():>8.3f}"
          f"{g('csd').mean():>7.1f}°{g('acc').mean():>7.1f}%{np.nanmean(ti):>8.3f}{g('n_samples').mean():>7.0f}")

group_stats = {'n_participants': int(n),
               'mean_mace': float(M.mean()), 'median_mace': float(np.median(M)),
               'mean_plv': float(P.mean()), 'mean_csd': float(Cd.mean()), 'mean_acc': float(A.mean()),
               'mean_fail_pct': float(np.mean([r['fail_pct'] for r in results]))}

print('\n--- Accuracy correlations across subjects ---')
if n >= 3 and np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, A); rs1, ps1 = spearmanr(SNR, A)
    group_stats.update({'acc_vs_snr_pearson_r': float(r1), 'acc_vs_snr_pearson_p': float(p1),
                        'acc_vs_snr_spearman_r': float(rs1), 'acc_vs_snr_spearman_p': float(ps1)})
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if n >= 3 and np.std(IT) > 0:
    r2, p2 = pearsonr(IT, A); rs2, ps2 = spearmanr(IT, A)
    group_stats.update({'acc_vs_itpc_pearson_r': float(r2), 'acc_vs_itpc_pearson_p': float(p2),
                        'acc_vs_itpc_spearman_r': float(rs2), 'acc_vs_itpc_spearman_p': float(ps2)})
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

out = {'model': 'PEAP',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': 0.0,
                  'gt_band': GT_BAND, 'gt_fir_numtaps': GT_FIR_NUMTAPS,
                  'peap_ar_order': PEAP_AR_ORDER, 'peap_context_samp': PEAP_CONTEXT_SAMP,
                  'peap_pad_samp': PEAP_PAD_SAMP, 'peap_fir_numtaps': PEAP_FIR_NUMTAPS,
                  'data_dir': STIM_DIR, 'split': 'chronological cal/test from npz split array',
                  'channel': 'best_ch_idx from npz', 'conditions': ['model'],
                  'bias_correction': False, 'gt': 'single FIR-175 (Butterworth GT dropped)',
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians',
                  'note': ('single-condition model; headline metrics under metrics_pooled / '
                           'metrics_per_timepoint / errors_per_timepoint_rad (analogue of BOB opt_*); '
                           'PEAP returns NaN on failure -> dropped and counted in fail_pct')},
       'group_stats': group_stats,
       'subjects': results}
outpath = os.path.join(OUTDIR, 'peap_theta_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nSaved -> {outpath}")
print("Per subject: metrics_pooled + per-timepoint metrics, per-timepoint errors (rad), SNR, ITPC, peak, fail_pct.")
print("Same data/GT/channel/test-split as BOB/EPN -> join on 'name'; PEAP.metrics_pooled <-> BOB.opt_metrics_pooled.")

PEAP THETA — group summary
  n=20 participants (>=10 test samples)
  MACE 52.5°±5.3   PLV 0.506   CSD 67.0°   Acc 70.8%
  mean failure rate 40.0%

--- Per-timepoint ---
   ms     MACE     PLV     CSD     Acc    ITPC      N
  300    90.0°   0.000  180.0°    0.0%   0.121      0
  400    90.0°   0.000  180.0°    0.0%   0.130      0
  500    54.5°   0.488   68.8°   69.7%   0.157     50
  600    52.1°   0.519   65.9°   71.1%   0.157     50
  700    51.0°   0.526   65.1°   71.7%   0.138     50

--- Accuracy correlations across subjects ---
  Acc vs SNR : r=+0.351 (p=0.1295)  rho=+0.365 (p=0.1131)
  Acc vs ITPC: r=+0.250 (p=0.2875)  rho=+0.116 (p=0.6269)

Saved -> /content/drive/MyDrive/RAIN_IRC/results/PEAP_onset/peap_theta_results.json
Per subject: metrics_pooled + per-timepoint metrics, per-timepoint errors (rad), SNR, ITPC, peak, fail_pct.
Same data/GT/channel/test-split as BOB/EPN -> join on 'name'; PEAP.metrics_pooled <-> BOB.opt_metrics_pooled.
